# GraphRAG

## Import packages

In [22]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import time
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  bert_score_metric,
)

## Disable warnings

In [23]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook.

## Import packages

In [24]:
env_variables = [
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
  'OPENROUTER_API_KEY',
  'CHROMA_API_KEY',
  'CHROMA_TENANT',
  'CHROMA_DATABASE',
  'CHROMA_COLLECTION_NAME',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [25]:
app = NeuroRAG(debug=True, use_flare=False)
app.compile()

## Evaluate RAG

### Load QA dataset

In [26]:
mediqa_df = pd.read_csv('../datasets/pubmed_summary_qa.csv')[:50]
mediqa_df

,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...
5,How does the auditory system respond to differ...,The auditory system's response to sound varies...
6,What is tonotopic organization in the auditory...,Tonotopic organization refers to the mapping o...
7,What brain regions are involved in language pr...,Language processing involves areas in the pref...
8,How does bilingualism affect language processi...,Bilingualism is associated with overlapping ac...
9,What is functional magnetic resonance imaging ...,Functional magnetic resonance imaging (fMRI) i...


### Load cached RAGs responses

In [27]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-neurorag-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache[CACHE_KEY].keys())

34

In [28]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []
generation_times = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache[CACHE_KEY]:
    start_time = time.perf_counter()
    cache[CACHE_KEY][question] = app.invoke(question)['generation']
    elapsed = time.perf_counter() - start_time
    generation_times.append(elapsed)

  predicted_answers.append(cache[CACHE_KEY][question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

if generation_times:
  print(f'Generation times (n={len(generation_times)}):')
  print(f'  Mean:   {sum(generation_times) / len(generation_times):.2f}s')
  print(f'  Median: {sorted(generation_times)[len(generation_times) // 2]:.2f}s')
  print(f'  Min:    {min(generation_times):.2f}s')
  print(f'  Max:    {max(generation_times):.2f}s')
  print(f'  Total:  {sum(generation_times):.2f}s')
else:
  print('All answers loaded from cache, no generation times recorded.')

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
print('cos_score', cos_score)
bleu_score = bleu_metric(expected_answers, predicted_answers)
print('bleu_score', bleu_score)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
print('rogue_1_score', rogue_1_score)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
print('rogue_l_score', rogue_l_score)
factscore_score = factscore_metric(expected_answers, predicted_answers)
print('factscore_score', factscore_score)
bert_score = bert_score_metric(expected_answers, predicted_answers)
print('bert_score', bert_score)

29it [00:00, 282.10it/s]

[2026-03-21 20:48:50.510] ---GENERATE STEP-BACK QUERY---
[2026-03-21 20:48:51.032] ---GENERATE SUBQUERIES---
[2026-03-21 20:48:51.714] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-21 20:48:52.083] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-03-21 20:48:52.084] ---ROUTE QUESTION---
[2026-03-21 20:48:52.085] ---GENERATE HYDE DOCUMENTS---
[2026-03-21 20:48:53.183][2026-03-21 20:48:53.183] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 3.20 seconds...
Too Many Requests, waiting for 3.20 seconds...
Too Many Requests, waiting for 3.20 seconds...
Too Many Requests, waiting for 3.20 seconds...
Too Many Requests, waiting for 51.20 seconds...
Too Many Requests, waiting for 51.20 seconds...
Too Many Requests, waiting for 51.20 seconds...


29it [00:12, 282.10it/s]

Too Many Requests, waiting for 409.60 seconds...
Too Many Requests, waiting for 409.60 seconds...
[2026-03-21 20:50:53.186] pub_med_retriever_node timed out
[2026-03-21 20:50:53.190] ---GRADE DOCUMENTs---
[2026-03-21 20:50:53.190] ---AFTER EXACT DEDUPLICATION: 11 documents---
[2026-03-21 20:50:53.193] ---BM25 TOP CANDIDATES: 10 documents---
[2026-03-21 20:50:54.640] ---FINAL DOCUMENTS NUMBER: 3---
[2026-03-21 20:50:54.640] ---ASSESS GRADED DOCUMENTS---
[2026-03-21 20:50:54.640] ---DECISION: GENERATE---
[2026-03-21 20:50:54.640] ---GENERATE---
[2026-03-21 20:51:18.695] ---GRADE GENERATION---
[2026-03-21 20:51:19.804] ---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---


35it [02:29,  5.69s/it] 

...

49it [24:16, 98.53s/it]

[2026-03-21 21:13:06.623] ---GRADE GENERATION---
[2026-03-21 21:13:06.633] ---GENERATE STEP-BACK QUERY---
[2026-03-21 21:13:07.124] ---GENERATE SUBQUERIES---
[2026-03-21 21:13:13.308] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-21 21:13:13.568] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-03-21 21:13:13.569] ---ROUTE QUESTION---
[2026-03-21 21:13:13.569] ---GENERATE HYDE DOCUMENTS---
[2026-03-21 21:13:15.334][2026-03-21 21:13:15.335] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-03-21 21:13:15.805] pub_med_retriever_node HTTP Error 429: Too Many Requests
Too Many Requests, waiting for 26214.40 seconds...
Too Many Requests, waiting for 26214.40 seconds...
[2026-03-21 21:15:15.338] pub_med_retriever_node timed out
[2026-03-21 21:15:15.341] ---GRADE DOCUMENTs---
[2026-03-21 21:15:15.341] ---AFTER EXACT DEDUPLICATION: 6 documents---
[2026-03-21 21:15:15.342] ---BM25 TOP CANDIDATES: 6 documents---
[2026-03-21 21:15:17.874] ---FINAL DOCUMENTS NUMBER: 3---
[20

50it [26:48, 32.16s/it] 

[2026-03-21 21:15:38.523] ---DECISION: GENERATION ADDRESSES QUESTION---
Generation times (n=16):
  Mean:   100.49s
  Median: 140.62s
  Min:    22.11s
  Max:    246.42s
  Total:  1607.91s


cos_score 0.7595144501125922
bleu_score 0.0123824177308289
rogue_1_score 0.3120454257024347
rogue_l_score 0.22781879323629403
factscore_score 0.18518097732427635
bert_score 0.12572027742862701
